In [ ]:
## Benchmark Forecasting

# We evaluate benchmark models separately for each store.
# Each store uses its own history to create 42-day forecasts, and the metrics are stored per store and per model.
#Benchmark model: Mean,Naive,Drift,Seasonal Naive,AutoARIMA

In [10]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from pmdarima import auto_arima
except ImportError:
    auto_arima = None

# ==========================================================
# Load data
# ==========================================================
processed_dir = Path("data/processed")
if not processed_dir.exists():
    processed_dir = Path.cwd().parent / "data" / "processed"

sales = pd.read_csv(processed_dir / "sales_clean.csv")
sales["date"] = pd.to_datetime(sales["date"])
sales = sales.sort_values(["store_id", "date"])

# ==========================================================
# Helper functions
# ==========================================================

def mae(y_true, y_pred):
    return np.mean(np.abs(y_true - y_pred))


def mape(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    mask = y_true != 0
    return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100


def build_forecasts(train_series, test_len):
    forecasts = {}

    train_series = pd.Series(train_series).astype(float)
    train_series = train_series.replace([np.inf, -np.inf], np.nan).dropna()

    if len(train_series) < 10:
        return {
            "Mean": np.repeat(np.nan, test_len),
            "Naive": np.repeat(np.nan, test_len),
            "Drift": np.repeat(np.nan, test_len),
            "Seasonal Naive": np.repeat(np.nan, test_len),
            "AutoARIMA": np.repeat(np.nan, test_len),
        }

    forecasts["Mean"] = np.full(test_len, train_series.mean())
    forecasts["Naive"] = np.full(test_len, train_series.iloc[-1])

    drift = np.arange(1, test_len + 1)
    forecasts["Drift"] = train_series.iloc[-1] + drift * (
        (train_series.iloc[-1] - train_series.iloc[0]) / (len(train_series) - 1)
    )

    forecasts["Seasonal Naive"] = np.array([
        train_series.iloc[-7 + (i % 7)] for i in range(test_len)
    ])

    if auto_arima is not None:
        try:
            model = auto_arima(
                train_series,
                seasonal=True,
                m=7,
                stepwise=True,
                suppress_warnings=True,
                error_action="ignore",
            )
            forecasts["AutoARIMA"] = model.predict(n_periods=test_len)
        except Exception:
            forecasts["AutoARIMA"] = np.repeat(np.nan, test_len)
    else:
        forecasts["AutoARIMA"] = np.repeat(np.nan, test_len)

    return forecasts

# ==========================================================
# Train / test split by store
# ==========================================================
# We use all rows before the 42-day holdout period as training.
# The forecast origin is the day before the first test date, so no extra day is removed.

forecast_horizon = 42
results = []

for store_id, store_df in sales.groupby("store_id"):
    store_df = store_df.sort_values("date").copy()
    store_df["sales"] = pd.to_numeric(store_df["sales"], errors="coerce")
    store_df = store_df.dropna(subset=["sales"])

    if len(store_df) < forecast_horizon + 5:
        continue

    train_series = store_df.iloc[:-forecast_horizon]["sales"].astype(float)
    test_series = store_df.iloc[-forecast_horizon:]["sales"].astype(float)
    test_dates = store_df.iloc[-forecast_horizon:]["date"]

    forecasts = build_forecasts(train_series, forecast_horizon)

    for model_name, pred in forecasts.items():
        pred = np.asarray(pred, dtype=float)
        if np.isnan(pred).any():
            continue

        results.append({
            "store_id": store_id,
            "model": model_name,
            "mae": mae(test_series.values, pred),
            "mape": mape(test_series.values, pred),
            "forecast_dates": json.dumps([d.strftime("%Y-%m-%d") for d in test_dates]),
            "actual_values": json.dumps([float(v) for v in test_series.values]),
            "forecast_values": json.dumps([float(v) for v in pred]),
        })

results_df = pd.DataFrame(results)
print(results_df.head())
print(f"\nTotal rows: {len(results_df)}")

# ==========================================================
# Save results
# ==========================================================
output_dir = Path("data/processed")
output_dir.mkdir(parents=True, exist_ok=True)

output_path = output_dir / "benchmark_results_per_store.csv"
results_df.to_csv(output_path, index=False)

print(f"Saved: {output_path}")

# ==========================================================
# Summary by model
# ==========================================================
summary_df = (
    results_df.groupby("model")[["mae", "mape"]]
    .mean()
    .sort_values("mae")
)
print(summary_df)


  store_id           model          mae        mape  \
0  store_1            Mean  1062.011176   12.494584   
1  store_1           Naive  3696.261905  100.000000   
2  store_1           Drift  3871.385628  103.971858   
3  store_1  Seasonal Naive  1383.833333   39.491304   
4  store_1       AutoARIMA  1039.439371   14.893060   

                                      forecast_dates  \
0  ["2015-06-08", "2015-06-09", "2015-06-10", "20...   
1  ["2015-06-08", "2015-06-09", "2015-06-10", "20...   
2  ["2015-06-08", "2015-06-09", "2015-06-10", "20...   
3  ["2015-06-08", "2015-06-09", "2015-06-10", "20...   
4  ["2015-06-08", "2015-06-09", "2015-06-10", "20...   

                                       actual_values  \
0  [4071.0, 4102.0, 3591.0, 3627.0, 3695.0, 4256....   
1  [4071.0, 4102.0, 3591.0, 3627.0, 3695.0, 4256....   
2  [4071.0, 4102.0, 3591.0, 3627.0, 3695.0, 4256....   
3  [4071.0, 4102.0, 3591.0, 3627.0, 3695.0, 4256....   
4  [4071.0, 4102.0, 3591.0, 3627.0, 3695.0, 4256....

In [ ]:
#The poor performance of the Naive and Drift baselines is mainly due to the data pattern around the test split. 
# The last training point is unusually low, while the first test point rises sharply, 
# so these simple methods struggle to adapt and produce large errors. 
# This is a characteristic of the series rather than a coding issue.

In [ ]:
# ==========================================================
# ETS Benchmark Forecasting
# ==========================================================
#Since autoarima takes a lot of time, we also make ETS to balance efficiency and accuracy
# This cell adds ETS results without rerunning AutoARIMA.
# It uses the same train/test split, metrics, and result format as the benchmark cell.

#Accordind to EDA STEP17,18, Exploratory analysis revealed a clear weekly seasonal pattern across the dataset. 
#Therefore, a seasonal period of seven days was adopted. 
#Although no strong global trend was observed after aggregation, individual stores may still exhibit local trends. 
#Hence, an additive trend component was retained and its suitability was validated through benchmark comparison.

from statsmodels.tsa.holtwinters import ExponentialSmoothing

ets_results = []

for store_id, store_df in sales.groupby("store_id"):
    store_df = store_df.sort_values("date").copy()
    store_df["sales"] = pd.to_numeric(store_df["sales"], errors="coerce")
    store_df = store_df.dropna(subset=["sales"])

    if len(store_df) < forecast_horizon + 14:
        continue

    train_series = store_df.iloc[:-forecast_horizon]["sales"].astype(float)
    test_series = store_df.iloc[-forecast_horizon:]["sales"].astype(float)
    test_dates = store_df.iloc[-forecast_horizon:]["date"]

    try:
        ets_model = ExponentialSmoothing(
            train_series,
            trend="add",
            seasonal="add",
            seasonal_periods=7,
            initialization_method="estimated"
        ).fit(optimized=True)

        pred = ets_model.forecast(forecast_horizon)
        pred = np.maximum(np.asarray(pred, dtype=float), 0)

    except Exception:
        continue

    ets_results.append({
        "store_id": store_id,
        "model": "ETS",
        "mae": mae(test_series.values, pred),
        "mape": mape(test_series.values, pred),
        "forecast_dates": json.dumps([d.strftime("%Y-%m-%d") for d in test_dates]),
        "actual_values": json.dumps([float(v) for v in test_series.values]),
        "forecast_values": json.dumps([float(v) for v in pred]),
    })

ets_results_df = pd.DataFrame(ets_results)

# Remove old ETS rows if this cell is rerun, then append the new ETS results
results_df = results_df[results_df["model"] != "ETS"].copy()
results_df = pd.concat([results_df, ets_results_df], ignore_index=True)

print(ets_results_df.head())
print(f"\nETS rows added: {len(ets_results_df)}")

# ==========================================================
# Save updated benchmark results
# ==========================================================
output_path = output_dir / "benchmark_results_per_store_with_ets.csv"
results_df.to_csv(output_path, index=False)

print(f"Saved: {output_path}")

# ETS results for all stores
ets_results_df = pd.DataFrame(ets_results)

print(ets_results_df)
print(f"\nNumber of stores forecasted: {len(ets_results_df)}")

# Save ETS results only
ets_output = output_dir / "ets_results_per_store.csv"
ets_results_df.to_csv(ets_output, index=False)

print(f"Saved: {ets_output}")

    store_id model          mae       mape  \
0    store_1   ETS   510.298111  13.600065   
1   store_10   ETS  1065.707677  17.424163   
2  store_100   ETS   927.404271  15.861444   
3  store_101   ETS   728.609107  15.887730   
4  store_102   ETS  1268.910181  19.891188   

                                      forecast_dates  \
0  ["2015-06-08", "2015-06-09", "2015-06-10", "20...   
1  ["2015-06-08", "2015-06-09", "2015-06-10", "20...   
2  ["2015-06-08", "2015-06-09", "2015-06-10", "20...   
3  ["2015-06-08", "2015-06-09", "2015-06-10", "20...   
4  ["2015-06-08", "2015-06-09", "2015-06-10", "20...   

                                       actual_values  \
0  [4071.0, 4102.0, 3591.0, 3627.0, 3695.0, 4256....   
1  [6177.0, 5109.0, 5569.0, 5911.0, 5757.0, 5787....   
2  [5664.0, 5522.0, 5406.0, 4741.0, 4815.0, 6133....   
3  [5406.0, 5001.0, 4332.0, 4467.0, 4461.0, 3241....   
4  [6772.0, 6905.0, 6577.0, 6005.0, 5346.0, 5808....   

                                     forecast_val

In [19]:
from pathlib import Path
import pandas as pd

# ============================================================
# File paths
# ============================================================

processed_dir = Path("data/processed")
raw_dir = Path("data/raw")

# ============================================================
# Load data
# ============================================================

results = pd.read_csv(processed_dir / "benchmark_results_per_store_with_ets.csv")
metadata = pd.read_csv(raw_dir / "metadata.csv")
# ============================================================
# Merge store type
# ============================================================

results = results.merge(
    metadata[["store_id", "store_type"]],
    on="store_id",
    how="left"
)

# ============================================================
# Average performance by Store Type and Model
# ============================================================

storetype_summary = (
    results
    .groupby(["store_type", "model"], as_index=False)
    .agg(
        Average_MAPE=("mape", "mean"),
        Average_MAE=("mae", "mean"),
        Number_of_Stores=("store_id", "nunique")
    )
)

# ============================================================
# Best model for each Store Type
# ============================================================

best_by_storetype = (
    storetype_summary
    .sort_values(["store_type", "Average_MAPE"])
    .groupby("store_type", as_index=False)
    .first()
)

# ============================================================
# Overall average performance
# ============================================================

overall_summary = (
    results
    .groupby("model", as_index=False)
    .agg(
        Average_MAPE=("mape", "mean"),
        Average_MAE=("mae", "mean"),
        Number_of_Stores=("store_id", "nunique")
    )
    .sort_values("Average_MAPE")
)

overall_best = overall_summary.head(1)

# ============================================================
# Print results
# ============================================================

print("=" * 60)
print("Best model for each Store Type")
print("=" * 60)
print(best_by_storetype)

print()

print("=" * 60)
print("Overall Best Model")
print("=" * 60)
print(overall_best)

# ============================================================
# Save results
# ============================================================

storetype_summary.to_csv(
    "average_performance_by_storetype.csv",
    index=False
)


overall_summary.to_csv(
    "overall_model_performance.csv",
    index=False
)



print("\nSaved:")
print("average_performance_by_storetype.csv")
print("overall_model_performance.csv")


Best model for each Store Type
  store_type model  Average_MAPE  Average_MAE  Number_of_Stores
0          a   ETS     16.518481  1013.350507               377
1          b   ETS     11.933989  1339.722205                11
2          c   ETS     18.607097  1032.297718                75
3          d   ETS     17.772124  1074.388701               213

Overall Best Model
  model  Average_MAPE  Average_MAE  Number_of_Stores
2   ETS     17.070615  1039.995869               676

Saved:
average_performance_by_storetype.csv
overall_model_performance.csv


In [17]:
# ============================================================
# Validation: Winner Count by Store
# ============================================================

# Find the best model (lowest MAPE) for each store
best_model_each_store = (
    results
    .sort_values(["store_id", "mape"])
    .groupby("store_id", as_index=False)
    .first()
)

# Count how many stores each model wins
winner_summary = (
    best_model_each_store
    .groupby("model", as_index=False)
    .agg(
        Stores_Won=("store_id", "count")
    )
    .sort_values("Stores_Won", ascending=False)
)

# Percentage of stores won
winner_summary["Win_Rate (%)"] = (
    winner_summary["Stores_Won"] /
    winner_summary["Stores_Won"].sum() * 100
).round(2)

print()
print("=" * 60)
print("Validation: Best Model by Individual Store")
print("=" * 60)
print(winner_summary)




Validation: Best Model by Individual Store
            model  Stores_Won  Win_Rate (%)
1             ETS         449         66.42
2            Mean         135         19.97
0       AutoARIMA          88         13.02
3  Seasonal Naive           4          0.59
